Cell: title + Step 0
# Profiling: `olist_customers_dataset` (bronze)

Source: `s3a://bronze/olist_customers_dataset/` (Parquet, all columns `StringType`)
Profiled: _2026-10-06_

## Step 0: Framing and hypotheses

**What one row means (hypothesis):** one row per *order-level customer record*, not per person.
Olist issues a new `customer_id` for every order; `customer_unique_id` identifies the actual person.

**Expected keys**
| Column | Expectation |
|---|---|
| `customer_id` | Unique, never null; primary key of this table |
| `customer_unique_id` | Not unique: repeats for customers with more than one order |

**Joins to**
- `orders.customer_id` (expected 1:1: one customer record per order)
- `geolocation.geolocation_zip_code_prefix` via `customer_zip_code_prefix` (N:M, geolocation has many rows per prefix)

**Downstream use**
- Silver: cleaned customer records
- Stage 4: SCD Type 2 customer dimension keyed on `customer_unique_id`. Changes of city/state across a person's records are the SCD2 input, so **nothing here should be deduped away before that is understood**.

**Decisions this profiling feeds:** primary key, dedup rule (if any), string normalisation for city/state, zip handling, SCD2 natural key.

## Step 1: Structure

| Check | Expected | Observed |
|---|---|---|
| Row count | ~99,441 (equal to `orders` row count) | 99,441 |
| Column count | 5 | 5 |
| Column names match the schema exactly (no BOM, no whitespace) | Yes | _TBD_ |
| All columns `string` in bronze | Yes | Yes |

**Notes:** _TBD_

In [ ]:
from src.conf.spark_session import get_spark_session

In [ ]:
spark = get_spark_session(app_name="explore")

# Then your read works in the next cell:
df =spark.read.parquet("s3a://bronze/olist_customers_dataset/")
df.show()

In [ ]:
df.printSchema()

## Step 2: Grain and keys

### 2a. Primary key: `customer_id`
| Check | Expected | Observed |
|---|---|---|
| `count()` vs `countDistinct(customer_id)` | Equal | Yes |
| Nulls / empty / whitespace-only in `customer_id` | 0 | 0 |

In [ ]:
df.createOrReplaceTempView("customers")

In [ ]:
spark.sql("SELECT count(*),count(distinct customer_id), count(distinct customer_unique_id) FROM customers").show(truncate=False)

In [ ]:
spark.sql("SELECT count(*) FROM customers where customer_id is null or trim(customer_id) = ''").show(truncate=False)

### 2b. Duplicates
| Check | Expected | Observed |
|---|---|---|
| Exact full-row duplicates | 0 | 0 |
| Duplicate `customer_id` with differing attributes (conflicts) | 0 | 0 |
| If conflicts exist: which columns differ? | n/a | n/a |

 Note: count(distinct a, b, c) skips rows with nulls. 
 Assumes no nulls in zip/city/state; confirm in Step 3.

In [ ]:
spark.sql("SELECT count(*),count(distinct *) FROM customers").show(truncate=False)

In [ ]:
spark.sql("SELECT customer_id,count(customer_id) as total_records FROM customers group by customer_id having count(customer_id) >1 ").show(truncate=False)

### 2c. Natural key for the person: `customer_unique_id`
| Check | Expected | Observed |
|---|---|---|
| `countDistinct(customer_unique_id)` | Less than row count | Yes |
| Nulls in `customer_unique_id` | 0 | 0 |
| Distribution of records per `customer_unique_id` (1, 2, 3, max) | Mostly 1, long tail | 17 records: 1 customer,9 records: 1 customers,7 records: 3 customers,6 records: 6 customers,5 records: 8 customers,4 records: 30 customers, 3 records : 203 customers, 2 records:2745 customers, 1 record : 93099 customers |
| Persons with >1 distinct `(zip, city, state)` across their records | Some: these are SCD2 changes | 252 |

**Look at examples:** 3–5 `customer_unique_id`s with multiple records, full rows side by side.

**Conclusion on grain:** 
- The Step 0 guess is confirmed: one row = one order's customer record.
- customer_id is unique (99,441 rows = 99,441 distinct).
- 96,096 people (customer_unique_id), mostly one record each: 93,099 have 1,max is 17.
- 252 people have more than one address (same result raw and normalized).
- Not yet checked: 1:1 match with orders.customer_id (needs the orders table).

Open for SCD2 (Stage 4): the customer address is the delivery address per order, so an "address change" may not be a move.

In [ ]:
spark.sql("SELECT count(*) FROM customers where customer_unique_id is null or trim(customer_unique_id) = ''").show(truncate=False)

In [ ]:
spark.sql("""
    SELECT
        total_rec,
        count(customer_unique_id) as total_customers
    FROM (
        SELECT
            customer_unique_id,
            count(*) as total_rec
        FROM customers
        GROUP BY customer_unique_id
    ) A
    GROUP BY total_rec
    ORDER BY total_rec desc
""").show(truncate=False)

In [ ]:
spark.sql("SELECT customer_unique_id,count(distinct customer_zip_code_prefix, customer_city, customer_state) as address_rec FROM customers group by customer_unique_id having address_rec > 1 order by address_rec desc").show(100, truncate=False)


In [ ]:
spark.sql("SELECT COUNT(*) FROM(SELECT customer_unique_id,count(distinct customer_zip_code_prefix, customer_city, customer_state) as address_rec FROM customers group by customer_unique_id having address_rec > 1 )A").show(100, truncate=False)


In [ ]:
# Normalized
spark.sql("SELECT COUNT(*) FROM(SELECT customer_unique_id,count(distinct trim(lower(customer_zip_code_prefix)), trim(lower(customer_city)), trim(lower(customer_state))) as address_rec FROM customers group by customer_unique_id having address_rec > 1)A").show(100, truncate=False)


In [ ]:
spark.sql("""
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state
FROM customers
where customer_unique_id in (
    '3e43e6105506432c953e165fb2acf44c',
    'd44ccec15f5f86d14d6a2cfa67da1975',
    '9832ae2f7d3e5fa4c7a1a06e9551bc61',
    'd130666a33d3681209c5c752fe56ee10',
    '67bddde1844b238fe25995cde1612b4e'
)
order by customer_unique_id
""").show(truncate=False)


  - 3e43e6105506432c953e165fb2acf44c: same city, 3 neighbouring zips. It looks like they ordered to two or three nearby addresses repeatedly (home and work, for example).
  - 67bddde1844b238fe25995cde1612b4e: same state, 2 cities. Ordered to 2 different cities.
  - 9832ae2f7d3e5fa4c7a1a06e9551bc61: same state, 2 cities. Ordering to 3 differnt zipcodes.
  - d130666a33d3681209c5c752fe56ee10: Same state, 2 cities. Ordered to 2 different cities.
  - d44ccec15f5f86d14d6a2cfa67da1975: 3 different states, far apart. A move, or orders delivered to different cities.

## Step 3: Column-level profiling

### Completeness (bronze is all strings, so missing can take several forms)
| Column | Nulls | Empty `''` | Whitespace-only | Placeholder values (`N/A`, `null`, `0`…) |
|---|---|---|---|---|
| `customer_id` | 0 | 0 | 0 | 0 |
| `customer_unique_id` | 0 | 0 | 0 | 0 |
| `customer_zip_code_prefix` | 0 | 0 | 0 | 0 |
| `customer_city` | 0 | 0 | 0 | 0 |
| `customer_state` | 0 | 0 | 0 | 0 |

### Cardinality
| Column | Distinct count | % of rows | Type of column (ID / categorical / free text) |
|---|---|---|--|
| `customer_id` | 99441 | 100% | ID: Every row different |
| `customer_unique_id` | 96096 | 96.6% | ID that repeats |
| `customer_zip_code_prefix` | 14994 | 15% | high-cardinality categorical |
| `customer_city` | 4119 | 4% | high-cardinality free text |
| `customer_state` | 27 | 0.03% | Categorical (low) |

### Validity
| Column | Rule | Violations (count / %) |
|---|---|---|
| `customer_id` | 32 lowercase hex chars | 0/ 0% |
| `customer_unique_id` | 32 lowercase hex chars | 0 /0% |
| `customer_zip_code_prefix` | Exactly 5 digits (leading zeros preserved) | 0 / 0% |
| `customer_state` | One of the 27 UFs: AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR PE PI RJ RN RS RO RR SC SP SE TO | 0 / 0% |
| `customer_city` | Consistent form: Lowercase a–z, space, hyphen, apostrophe | 0.001% (quilometro 14 do mutum, a valid name with digits)|

451 fail strict a–z: 224 hyphen, 226 apostrophe, 1 digits. All valid names.

### Distribution
| Check | Observed |
|---|---|
| Top 10 states by row count (share %) | States: SP 42.0%, RJ 12.9%, MG 11.7%; top 3 = 66.6%; all others ≤ 5.5%  |
| Top 10 cities by row count (share %) | sao paulo 15.6%, rio de janeiro 6.9%, belo horizonte 2.8%; others ≤ 2.1% |
| Zip prefixes: min / max | 01003/99990 |
| Any single value dominating (join/partition skew risk)? | high for state/city keys. Our joins use customer_id and SCD2 uses customer_unique_id, both evenly spread, so no impact on the pipeline. Don't use customer_state as a write-partition column.  No practical impact at 99k rows. |

**Look at examples:** offending rows for each validity rule that has violations.


### Completeness 

In [ ]:
spark.sql("""
SELECT
    count_if(TRIM(customer_unique_id) ='' OR  customer_unique_id IS NULL) AS null_empty_cust_unique_id_count,
    count_if(TRIM(customer_id) ='' OR  customer_id IS NULL) AS null_empty_cust_id_count,
    count_if(TRIM(customer_zip_code_prefix) ='' OR  customer_zip_code_prefix IS NULL) AS null_empty_cust_zip_code_prefix_count,
    count_if(TRIM(customer_city) ='' OR  customer_city IS NULL) AS null_empty_cust_city_count,
    count_if(TRIM(customer_state) ='' OR  customer_state IS NULL) AS null_empty_cust_state_count
FROM customers

""").show(truncate=False)

In [ ]:
spark.sql("""
SELECT
    count_if(LOWER(TRIM(customer_unique_id)) in ('null', 'n/a', '0','none')) AS placeholder_cust_unique_id_count,
    count_if(LOWER(TRIM(customer_id)) in ('null', 'n/a', '0', 'none')) AS placeholder_cust_id_count,
    count_if(LOWER(TRIM(customer_zip_code_prefix)) in ('null', 'n/a', '0', 'none') OR LOWER(TRIM(customer_zip_code_prefix)) = '00000') AS placeholder_cust_zip_code_prefix_count,
    count_if(LOWER(TRIM(customer_city)) in ('null', 'n/a', '0', 'none')) AS placeholder_cust_city_count,
    count_if(LOWER(TRIM(customer_state)) in ('null', 'n/a', '0', 'none')) AS placeholder_cust_state_count
FROM customers
""").show(truncate=False)

### Cardinality

In [ ]:
spark.sql("""
SELECT
    count(distinct customer_unique_id) AS distinct_cust_unique_id_count,
    count(distinct customer_id) AS distinct_cust_id_count,
    count(distinct customer_zip_code_prefix) AS distinct_cust_zip_code_prefix_count,
    count(distinct customer_city) AS distinct_cust_city_count,
    count(distinct customer_state) AS distinct_cust_state_count
FROM customers
""").show(truncate=False)

### Validity

In [ ]:
spark.sql("""
SELECT
    count_if(NOT (customer_zip_code_prefix rlike '^[0-9]{5}$')) AS invalid_cust_zip_code_prefix_count
FROM customers
""").show(truncate=False)

In [ ]:
spark.sql("""
SELECT
    count_if(NOT customer_city rlike "^[a-z '-]+$") AS invalid_cust_city_count
FROM customers
""").show(truncate=False)

In [ ]:
spark.sql("""
SELECT
    'mogi-guacu' RLIKE "^[a-z '-]+$",
    "dias d'avila" RLIKE "^[a-z '-]+$",
        'são paulo'    RLIKE "^[a-z '-]+$"    -- false (ã isn't allowed)
""").show(truncate=False)

In [ ]:
spark.sql("""
SELECT
    count_if(NOT (customer_city rlike "^[a-z ]+$")) AS invalid_cust_city_with_out_alphabets_and_spaces_count,
    count_if(NOT (customer_city rlike "^[a-z -]+$")) AS invalid_cust_city_with_out_hyphens_count,
    count_if(NOT (customer_city rlike "^[a-z '-]+$")) AS invalid_cust_city_with_out_apostrophes_count
FROM customers
""").show(50, truncate=False)

In [ ]:
spark.sql("""
SELECT
    *
FROM customers
WHERE NOT (customer_city rlike "^[a-z '-]+$")
""").show(50, truncate=False)

In [ ]:
spark.sql("""
SELECT
    customer_city,
    count(*)
FROM customers
WHERE NOT (customer_city rlike "^[a-z '-]+$")
group by customer_city
order by count(*) desc
""").show(50, truncate=False)

In [ ]:
spark.sql("""
SELECT
    count(distinct customer_state)
FROM customers
""").show(truncate=False)

In [ ]:
spark.sql("""
SELECT
    count_if(NOT (customer_state rlike '^[A-Z]{2}$')) AS invalid_cust_state_count
FROM customers
""").show(truncate=False)

In [ ]:
spark.sql("SELECT distinct TRIM(customer_state) as trim_state,  customer_state as state FROM customers").show(30, truncate=False)

the two IDs: exactly 32 lowercase hex characters (rlike)

In [ ]:
spark.sql("""
SELECT
    count_if(NOT (customer_id rlike '^[a-f0-9]{32}$')) AS invalid_cust_id_count,
    count_if(NOT (customer_unique_id rlike '^[a-f0-9]{32}$')) AS invalid_cust_unique_id_count
FROM customers
""").show(truncate=False)

### Distribution

In [ ]:
spark.sql("""
SELECT
    customer_state,
    round((count(*) * 100.0)/(sum(count(*)) OVER ()), 2) AS pct_of_rows
FROM customers
GROUP BY customer_state
ORDER BY pct_of_rows DESC
LIMIT 10
""").show(truncate=False)

In [ ]:
spark.sql("""
SELECT
    customer_city,
    round((count(*) * 100.0)/(sum(count(*)) OVER ()), 2) AS pct_of_rows
FROM customers
GROUP BY customer_city
ORDER BY pct_of_rows DESC
LIMIT 10
""").show(truncate=False)

In [ ]:
spark.sql("""
SELECT
    max(customer_zip_code_prefix) AS max_cust_zip_code_prefix,
    min(customer_zip_code_prefix) AS min_cust_zip_code_prefix
FROM customers
""").show(truncate=False)

Cell: Step 4
## Step 4: Cross-column consistency (within a row)

| Check | Expected | Observed |
|---|---|---|
| Same `customer_city` appears under more than one `customer_state` | Rare (genuine homonym cities exist in Brazil) | _TBD_ |
| Same `customer_zip_code_prefix` maps to more than one `customer_state` | 0: a zip prefix belongs to one state | _TBD_ |
| Same `customer_zip_code_prefix` maps to more than one `customer_city` | Some: one prefix can cover several municipalities | _TBD_ |
| City name variants that are the same city (accents/spaces/case) | Some | _TBD_ |

**Look at examples:** for every non-zero result, list the actual values.

---

## Findings → decisions

| # | Finding | Impact (count / %) | Decision (fix / drop / quarantine / flag / accept) | Blocking? | Layer | Test |
|---|---|---|---|---|---|---|
| 1 | _TBD_ | _TBD_ | _TBD_ | _TBD_ | silver | _TBD_ |

---